In [ ]:
!pip install emoji

In [ ]:
import pandas as pd
import torch
import re
import emoji
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments

In [ ]:
df = pd.read_csv("/content/Tweets.csv")
#EDA
display(df.info())
df.shape


In [ ]:
df[['airline_sentiment', 'text']].isnull().sum()

In [ ]:
df['airline_sentiment'].value_counts()

In [ ]:
#keep only text and sentiment because just that what we need
df = df[['text', 'airline_sentiment']]

#keep only positive and negative
df = df[df['airline_sentiment'] != 'neutral']

df['airline_sentiment'].value_counts()

In [ ]:
sns.countplot(x='airline_sentiment', data=df)

In [ ]:
df.shape

In [ ]:
df['text_length'] = df['text'].apply(len)
df['text_length'].describe()

In [ ]:
#Encoding negative=0, positive=1
df['labels'] = df['airline_sentiment'].map({'negative': 0, 'positive': 1})

#final dataset
df = df[['text', 'labels']]
display(df.head())
df['labels'].value_counts()

In [ ]:
#data split train 80% /test 20%
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['labels'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['labels']
)

In [ ]:
#preprocessing and data cleaning
slang_dict = {
    "omg": "oh my god",
    "lol": "laughing out loud",
    "u": "you",
    "r": "are",
    "btw": "by the way",
    "idk": "i do not know",
    "smh": "shaking my head",
    "tbh": "to be honest",
}
def expand_slang(text):
    words = text.split()
    new_words = []
    for w in words:
        key = w.lower()
        if key in slang_dict:
            new_words.append(slang_dict[key])
        else:
            new_words.append(w)
    return " ".join(new_words)


def preprocess_tweet(text):
    text = text.lower()
    text = emoji.demojize(text)
    text = expand_slang(text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)
    text = re.sub(r'(.)\1{2,}', r'\1', text)
    text = re.sub(r"[^a-zA-Z0-9\s:]", "", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [ ]:
#apply preprocessing learned from training data
train_texts = [preprocess_tweet(t) for t in train_texts]
test_texts  = [preprocess_tweet(t) for t in test_texts]

In [ ]:
#tokenization
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)


In [ ]:
#creating a dataset for BERT training (prepare it for the model)
class TwitterDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key,val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_dataset = TwitterDataset(train_encodings, train_labels)
test_dataset = TwitterDataset(test_encodings, test_labels)

In [ ]:
#loading the model
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
#setting training parameters for our model
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    report_to="none"
)

In [ ]:
#how to mesure model performance
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    return {"accuracy": acc, "f1": f1}

In [ ]:
#trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
print("Evaluation results:", trainer.evaluate())

In [ ]:
def predict_sentiment(tweets):
    inputs = tokenizer(tweets, return_tensors="pt", padding=True, truncation=True)
    inputs = {key: val.to(device) for key,val in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=1)
    labels = ["negative","positive"]
    return [labels[p.item()] for p in predictions]

In [ ]:
device = torch.device("cpu")
model.to(device)

In [ ]:
new_tweets = ["i caaaaaaan't this is my best triiiiiip ever​", "👌​","i don't really like it "]
new_tweets_clean = [preprocess_tweet(t) for t in new_tweets]
predictions = predict_sentiment(new_tweets_clean)
for t,p in zip(new_tweets, predictions):
    print(t,"→",p)